# 03 - Mine binding cliffs

A **binding cliff** is a pair of records whose varied sequence (peptide or CDR3b)
is within `k` edits but whose binding outcome flips, with the context (the other
chain + MHC) held fixed. We detect neighbour pairs, summarise them, build a cliff
graph, and show a concrete single-residue flip family.

In [ ]:
from tcr_cliff.data import load_toy
from tcr_cliff.config import CliffConfig
from tcr_cliff.cliffs import (
    find_neighbor_pairs,
    cliff_statistics,
    cliff_pairs_to_frame,
    build_cliff_graph,
    cliff_components,
    cliff_hubs,
)

df = load_toy()

# k=1 edit, vary either chain, require a binding label flip (the cliff definition).
cliff_cfg = CliffConfig(max_edits=1, vary='both', require_label_flip=True, same_context=True)
pairs = find_neighbor_pairs(df, cliff_cfg)
print(f'{len(pairs)} neighbour pairs total')

## Cliff statistics

The `cliff_ratio` is the dataset's ruggedness.

In [ ]:
stats = cliff_statistics(pairs, n_records=len(df))
for key in (
    'n_neighbor_pairs', 'n_cliff_pairs', 'n_smooth_pairs', 'cliff_ratio',
    'n_records_in_cliff', 'cliffs_by_vary', 'cliffs_by_distance',
):
    print(f'{key:22s}: {stats[key]}')

## Inspect the detected pairs as a table

In [ ]:
pf = cliff_pairs_to_frame(pairs)
cols = ['vary', 'seq_i', 'seq_j', 'distance', 'label_i', 'label_j', 'is_cliff', 'reason']
pf.loc[pf['is_cliff'], cols].head(8)

## A concrete single-residue flip family

Pick a peptide cliff and line up the two sequences to see exactly which residue
position flips the binding label.

In [ ]:
cliffs = [p for p in pairs if p.is_cliff and p.vary == 'peptide']
p = cliffs[0]
print('context (CDR3b held fixed):', df.loc[p.i, 'cdr3b'])
print('MHC                       :', df.loc[p.i, 'mhc'])
print()
print(f'{p.seq_i}  binder={p.label_i}')
print(f'{p.seq_j}  binder={p.label_j}')
marker = ''.join('^' if a != b else ' ' for a, b in zip(p.seq_i, p.seq_j))
print(marker, ' <- the single residue that flips binding')

## Build and inspect the cliff graph

Nodes are records; edges are cliff (or smooth) neighbour relations. Connected
components are *cliff families*; hubs are records on many cliffs.

In [ ]:
g = build_cliff_graph(pairs, df=df, cliffs_only=True)
print('cliff graph:', g.number_of_nodes(), 'nodes,', g.number_of_edges(), 'edges')

comps = cliff_components(g)
print('n cliff families (components):', len(comps))
print('largest family size          :', max((len(c) for c in comps), default=0))

hubs = cliff_hubs(g, top=5)
print('top cliff hubs:', hubs)

### Draw it (optional)
Guarded so it never hard-fails if matplotlib is unavailable.

In [ ]:
try:
    import matplotlib
    matplotlib.use('Agg')  # headless; no plt.show()
    import matplotlib.pyplot as plt
    import networkx as nx

    sub = g.subgraph(max(comps, key=len)) if comps else g
    fig, ax = plt.subplots(figsize=(5, 4))
    pos = nx.spring_layout(sub, seed=0)
    nx.draw(sub, pos, ax=ax, node_size=120, with_labels=False, edge_color='crimson')
    ax.set_title('Largest cliff family')
    fig.savefig('cliff_family.png', dpi=110, bbox_inches='tight')
    plt.close(fig)
    print('saved cliff_family.png')
except Exception as exc:  # pragma: no cover - plotting is optional
    print('plotting skipped:', exc)

### Next
Continue to **04_baseline_vs_cliffaware** for the headline benchmark.